In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

## Langsmith Tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Groq
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
# Data Ingestion

import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://www.indiatoday.in/world/story/pakistan-afghanistan-conflict-pak-bombs-kabul-taliban-soldiers-killed-major-cross-border-escalation-2875134-2026-02-27",),
)
documents = loader.load()
print(documents)

e:\Langchain\langchain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.indiatoday.in/world/story/pakistan-afghanistan-conflict-pak-bombs-kabul-taliban-soldiers-killed-major-cross-border-escalation-2875134-2026-02-27', 'title': 'Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today', 'description': 'Tensions sharply escalated as Pakistan launched airstrikes in multiple Afghan cities targeting Taliban installations, hours after Kabul claimed cross-border attacks, marking one of the most serious flare-ups along the Durand Line.\r\n', 'language': 'en'}, page_content='Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today India TodayAaj TakIndia Today HindiNewsTakGNTTVLallantopBusiness TodayBanglaMalayalamNortheastBT BazaarHarper\'s BazaarSports TakCrime TakAstro TakGamingBrides TodayCosmopolitanKisan TakIshq FMReader’s DigestIndia TodayAaj TakIndia Today HindiNewsTakGNTT

In [3]:
# Chunking
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

print(docs)

[Document(metadata={'source': 'https://www.indiatoday.in/world/story/pakistan-afghanistan-conflict-pak-bombs-kabul-taliban-soldiers-killed-major-cross-border-escalation-2875134-2026-02-27', 'title': 'Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today', 'description': 'Tensions sharply escalated as Pakistan launched airstrikes in multiple Afghan cities targeting Taliban installations, hours after Kabul claimed cross-border attacks, marking one of the most serious flare-ups along the Durand Line.\r\n', 'language': 'en'}, page_content="Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today India TodayAaj TakIndia Today HindiNewsTakGNTTVLallantopBusiness TodayBanglaMalayalamNortheastBT BazaarHarper's BazaarSports TakCrime TakAstro TakGamingBrides TodayCosmopolitanKisan TakIshq FMReader’s DigestIndia TodayAaj TakIndia Today HindiNewsTakGNTTV

In [4]:
docs

[Document(metadata={'source': 'https://www.indiatoday.in/world/story/pakistan-afghanistan-conflict-pak-bombs-kabul-taliban-soldiers-killed-major-cross-border-escalation-2875134-2026-02-27', 'title': 'Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today', 'description': 'Tensions sharply escalated as Pakistan launched airstrikes in multiple Afghan cities targeting Taliban installations, hours after Kabul claimed cross-border attacks, marking one of the most serious flare-ups along the Durand Line.\r\n', 'language': 'en'}, page_content="Pakistan-Afghanistan conflict: Pak bombs Kabul, Taliban claims 55 soldiers killed in major cross-border escalation - India Today India TodayAaj TakIndia Today HindiNewsTakGNTTVLallantopBusiness TodayBanglaMalayalamNortheastBT BazaarHarper's BazaarSports TakCrime TakAstro TakGamingBrides TodayCosmopolitanKisan TakIshq FMReader’s DigestIndia TodayAaj TakIndia Today HindiNewsTakGNTTV

In [5]:
# embeddings 
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma


embeddings =  OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)

vectordb = Chroma.from_documents(documents=docs, embedding=embeddings,persist_directory="./india-today-db")

query = "PLAYER OF THE MATCH ?"
docs = vectordb.similarity_search(query)
docs[0].page_content

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_1896\2488538545.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings =  OllamaEmbeddings(


'Follow Us On:                      Advertisement\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0PUBLICATIONSIndia TodayBusiness TodayIndia Today-HindiTIME TELEVISIONIndia Today TVAaj Tak Good News TodayEVENTSAgenda AajTakIndia Today ConclaveSahitya AajTak RADIOIshq FMAajTak RadioGAMINGIndia Today Gaming World Esports CupUSEFUL LINKSPress ReleaseSitemapNewsNewsletter Privacy PolicyCorrection PolicyLMIL DocumentsPRINTINGThomson PressWELFARECare TodayDISTRIBUTIONRate Card    SYNDICATIONS  Headlines Today        WEBSITES  India Today India Today Malayalam India Today NE Business Today DailyO AajTak Lallantop Bangla GNTTV iChowk Reader’s Digest Cosmopolitan    EDUCATION  Vasant Valley Best Colleges Best Universities          TRENDING TOPICST20 World Cup Points TableT20 World Cup ScheduleT20 World CupAssam Election ConstituenciesWest Bengal Election ConstituenciesPuducherry Election ConstituenciesTamil Nadu Election ConstituenciesKerala Election ConstituenciesDelhi PollutionAir Quality IndexMumbai AQINoida

In [6]:
## Retrieval Chain, Documents Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama

# 5. LLM + prompt for generation
# llm = Ollama(model="gemma3:1b")
from langchain_groq import ChatGroq

llm=ChatGroq(model="qwen/qwen3-32b" , groq_api_key=groq_api_key)


prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    
    """
)

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    \n    '), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021666D52560>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021666E894

In [7]:
## input --> retriver --> vectorstoredb

retriever=vectordb.as_retriever()

from langchain_classic.chains.retrieval import  create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever , document_chain)

In [8]:
# Get the Response from LLM

response = retrieval_chain.invoke({"input":"Pakistan in 'open war' with Afghanistan, Minister declares as conflict escalates"})

print(response['answer'])
print(response)


<think>
Okay, let's tackle this question. The user wants to know if Pakistan and Afghanistan are in an "open war" according to the provided context. First, I need to look through the context to find evidence supporting that.

Looking at the text, there's a mention of Pakistan's Defence Minister, Khawaja Asif, declaring that Pakistan is in an "open war" with Afghanistan. That seems like a direct statement. Also, there are details about airstrikes by Pakistan in Afghan cities and retaliatory strikes from the Taliban. Both sides are exchanging fire, and there are casualty reports from each side. The term "open war" is used in the minister's quote, which is a strong indicator. Additionally, the article refers to this as one of the most serious flare-ups in years, indicating a significant escalation. The conflicting casualty figures and the denial about soldiers being captured also suggest active hostilities. 

I need to make sure there's no conflicting information in the context. The key p

In [10]:
response = retrieval_chain.invoke({"input":"node js code"})

print(response['answer'])

<think>
Okay, let's see. The user is asking a question based on the provided context. The context is about Pakistan and Afghanistan conflict, specifically mentioning Pakistan bombing Kabul and the Taliban claiming 55 soldiers killed. The user hasn't provided the actual question yet. But the task is to answer the question based only on the context.

Wait, the user might have forgotten to include the specific question. The current prompt just says "Answer the following question based only on the provided context:" but there's no question after that. Hmm. Let me check again. The context includes information about Pakistan-Afghanistan conflict, airstrikes, casualties, and mentions of the Durand Line. Maybe the original question was supposed to be something like "What event caused the escalation between Pakistan and Afghanistan?" or "How many soldiers did the Taliban claim were killed?" 

But since the user hasn't provided the actual question, I can't proceed. However, looking at the struct

In [ ]:
print(response['context'])


[Document(id='9de8ca2d-4ed2-4911-85fa-61e4c8b80b23', metadata={'title': 'Cricket commentary | India vs Pakistan, 12th Match, ICC Cricket World Cup 2023', 'language': 'en', 'description': 'Follow IND 192/3 (30.3) vs PAK\n                191 (Shreyas Iyer 53(62) KL Rahul 19(29)) | India vs Pakistan, 12th Match, ICC Cricket World Cup 2023, Sat, Oct 14, ICC Cricket World Cup 2023 with live Cricket score, ball by ball commentary updates on Cricbuzz', 'source': 'https://www.cricbuzz.com/live-cricket-scores/75476/ind-vs-pak-12th-match-icc-cricket-world-cup-2023'}, page_content='Top\xa0APPSAndroidiOSFOLLOW US ONFacebookTwitterYoutubePinterestCOMPANYCareersAdvertiseCricbuzz TV AdsAbout UsPrivacy NoticeTerms of Service© 2026 Cricbuzz.com, Cricbuzz Platforms Limited. All rights reserved | The Times of India | Navbharat Times'), Document(id='952b4e4b-c304-4618-bf80-a78d43d4dab3', metadata={'description': 'Follow IND 192/3 (30.3) vs PAK\n                191 (Shreyas Iyer 53(62) KL Rahul 19(29)) | I